In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os

images_path=os.path.join(path, 'dataset', 'images')
masks_path=os.path.join(path, 'dataset', 'masks')

print(images_path)
print(masks_path)

In [ ]:
# TO DO

from torch.utils.data import Dataset
from PIL import Image
class segementationdataset(Dataset):
  def __init__(self,transform=None, target_transform=None):
    self.path=path
    self.transform=transform
    self.target_transform=target_transform
    self.image_paths=[]
    self.masks=[]

    for filename in os.listdir(self.path):
      img_path=os.path.join(self.path,filename)
      mask_path=os.path.join(self.path, filename)
      self.image_path.append(img_path)
      self.masks.append(mask_path)


    def __len__(self):
      return len(self.image_paths)

    def __getitem__(self, index):
      img=self.image_paths[index]
      mask=self.masks[index]

      image=Image.open(img).convert('RGB')

      if self.transform:
        image=self.transform(image)

      if self.target_transform:
        mask=self.transform(mask)

      return image, mask


In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",  #Pretrained encoder (backbone)
    encoder_weights="imagenet",
    in_channels=3,  # RGB images
    classes=8,
    activation="sigmoid"  # Apply Sigmoid activation directly in the model
).to(device)

In [ ]:
# TO DO



In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

#Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

#Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO


In [ ]:
import torch.optim as optim
import torch.nn as nn
# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# TO DO